### Importing Packages

In [ ]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import nltk
import pandas as pd
import numpy as np

### Load Dataset

In [ ]:
airline = pd.read_parquet('airline_clean.parquet', engine='pyarrow')
airline.dtypes

In [ ]:
airline.head(1)

In [ ]:
airline['airline_num'] = airline['airline'].astype('category').cat.codes

In [ ]:
airline['day_num'] = airline['day_of_week']
airline[['airline', 'airline_num', 'day_num']].drop_duplicates().head()

In [ ]:
airline['tweet_num'] = airline.groupby(['airline_num', 'day_num']).cumcount()
airline[['airline_num', 'day_num', 'tweet_num']].head()

In [ ]:
LIB = (airline
    .groupby(['airline_num', 'airline'])
    .agg(
        total_tweets  = ('cleaned_text', 'count'),
        avg_length    = ('str_len', 'mean'),
        total_retweets= ('retweet_count', 'sum'),
        date_start    = ('tweet_created', 'min'),
        date_end      = ('tweet_created', 'max'),
    )
    .round(2)
    .reset_index()
    .set_index('airline_num')
    .sort_index()
)
print(LIB[['airline', 'avg_length']])

In [ ]:
LIB

In [ ]:
LIB.to_csv('LIB.csv')

### Tokenization

In [ ]:
from nltk.tokenize import sent_tokenize

SENTS = (airline
    .set_index(['airline_num', 'day_num', 'tweet_num'])['cleaned_text']
    .apply(lambda x: sent_tokenize(str(x)))
    .explode()
    .to_frame('sent_str')
    .reset_index()
)
SENTS['sent_num'] = SENTS.groupby(['airline_num', 'day_num', 'tweet_num']).cumcount()
SENTS = SENTS.set_index(['airline_num', 'day_num', 'tweet_num', 'sent_num'])
SENTS.head()

In [ ]:
TOKENS = (SENTS['sent_str']
    .apply(lambda x: word_tokenize(str(x)))
    .explode()
    .to_frame('token_str')
    .reset_index()
)

TOKENS['token_num'] = TOKENS.groupby(
    ['airline_num', 'day_num', 'tweet_num', 'sent_num']
).cumcount()

TOKENS = TOKENS.set_index(
    ['airline_num', 'day_num', 'tweet_num', 'sent_num', 'token_num']
)
TOKENS

### POS Tagging

In [ ]:
from nltk.corpus import wordnet

def tag_sentence(group):
    tagged = nltk.pos_tag(group['token_str'].tolist())
    group = group.copy()
    group['pos'] = [tag for _, tag in tagged]
    return group

TOKENS = (TOKENS
    .reset_index()
    .groupby(['airline_num', 'day_num', 'tweet_num', 'sent_num'], group_keys=False)
    .apply(tag_sentence)
    .set_index(['airline_num', 'day_num', 'tweet_num', 'sent_num', 'token_num'])
)
TOKENS.head(10)

### Lemmatization

In [ ]:
def get_wordnet_pos(tag):
    if tag.startswith('J'): return wordnet.ADJ
    if tag.startswith('V'): return wordnet.VERB
    if tag.startswith('N'): return wordnet.NOUN
    if tag.startswith('R'): return wordnet.ADV
    return wordnet.NOUN

lemmatizer = WordNetLemmatizer()
TOKENS['lemma'] = TOKENS.apply(
    lambda row: lemmatizer.lemmatize(row['token_str'], get_wordnet_pos(row['pos'])),
    axis=1
)
TOKENS.head(10)

In [ ]:
TOKENS['term_str'] = TOKENS['lemma']

def get_pos_group(tag):
    if tag.startswith('NN'):                    return 'NOUN'
    if tag.startswith('VB') or tag == 'MD':     return 'VERB'
    if tag.startswith('JJ'):                    return 'ADJ'
    if tag.startswith('RB'):                    return 'ADV'
    if tag.startswith('PRP') or tag.startswith('WP'): return 'PRON'
    if tag in ('DT', 'WDT', 'PDT'):             return 'DET'
    if tag in ('IN', 'TO'):                     return 'PREP'
    if tag == 'CC':                             return 'CONJ'
    if tag == 'CD':                             return 'NUM'
    return 'OTHER'

TOKENS['pos_group'] = TOKENS['pos'].apply(get_pos_group)
TOKENS.head(10)

In [ ]:
import string, re

TOKENS = TOKENS[~TOKENS['token_str'].apply(lambda x: all(c in string.punctuation for c in str(x)))]

# Remove: pure numbers, colons from demojized emojis (:smile:), 
# single characters, and anything not purely alphabetic/hyphenated
TOKENS = TOKENS[TOKENS['token_str'].str.match(r'^[a-zA-Z][a-zA-Z\-]{1,}$')]

print(f"TOKENS after cleaning: {len(TOKENS):,}")
TOKENS.head()

### Remove stop words

In [ ]:
stop_words = set(stopwords.words('english'))
TOKENS['is_stop'] = TOKENS['token_str'].isin(stop_words)
TOKENS.head(10)

In [ ]:
print(f"Observations : {len(TOKENS)}")
print(f"OHCO: airline_num,day_num,tweet_num,sent_num,token_num")
print(f"Columns: {','.join(TOKENS.reset_index().columns.tolist())}")

TOKENS.to_csv('TOKENS.csv')

### VOCAB
The unique word types (terms) in the corpus.

In [ ]:
from nltk.stem import PorterStemmer
import numpy as np

ps = PorterStemmer()
tok = TOKENS.reset_index()

# Core counts per term
VOCAB = (TOKENS
    .groupby('term_str')
    .agg(
        n             = ('term_str', 'count'),
        max_pos       = ('pos',       lambda x: x.value_counts().idxmax()),
        max_pos_group = ('pos_group', lambda x: x.value_counts().idxmax()),
        stop          = ('is_stop',   'max'),
    )
)

# Probability and self-information
N = VOCAB['n'].sum()
VOCAB['p'] = VOCAB['n'] / N
VOCAB['i'] = -np.log2(VOCAB['p'])

# Document frequency unique tweets containing each term
N_docs = tok[['airline_num','day_num','tweet_num']].drop_duplicates().shape[0]
df_series = (tok[['airline_num','day_num','tweet_num','term_str']]
    .drop_duplicates()
    .groupby('term_str')
    .size()
    .rename('df')
)
VOCAB = VOCAB.join(df_series)
VOCAB['idf']   = np.log(N_docs / VOCAB['df'])
VOCAB['dfidf'] = VOCAB['df'] * VOCAB['idf']

# Porter stem
VOCAB['porter_stem'] = VOCAB.index.map(ps.stem)

# Ngram length
VOCAB['ngram_length'] = VOCAB.index.map(lambda x: len(x.split()))

VOCAB = VOCAB.sort_values('dfidf', ascending=False)
print(f"VOCAB size: {len(VOCAB):,}")
VOCAB.head()

In [ ]:
# Top 20 significant words by DFIDF
VOCAB[~VOCAB['stop']].sort_values('dfidf', ascending=False).head(20)[
    ['n','p','i','df','dfidf','porter_stem','max_pos','max_pos_group','stop']
]

In [ ]:
VOCAB.to_csv('VOCAB.csv')
print(f"Observations: {len(VOCAB)}")
print(f"Columns: {','.join(VOCAB.reset_index().columns.tolist())}")



## Bag of Words, DTM & TF-IDF

In [ ]:
MIN_FREQ     = 5                              # min corpus frequency to include
REMOVE_STOP  = True                           # drop stopwords
POS_FILTER   = ['NOUN', 'VERB', 'ADJ', 'ADV']# None = keep all POS groups
TOP_DFIDF    = None                           # int = top N by DFIDF, None = all
DOC_LEVEL    = ['airline_num','day_num','tweet_num'] 
N_COMPONENTS = 20                             # SVD components for reduced TFIDF

### BOW

In [ ]:
SIG_VOCAB = VOCAB[
    (~VOCAB['stop']) &
    (VOCAB['max_pos_group'].isin(POS_FILTER)) &
    (VOCAB['n'] >= MIN_FREQ) &
    (VOCAB.index.str.match(r'^[a-z][a-z\-]+$'))   # alphabetic terms only, no punctuation/emoji artifacts
].copy()

print(f"SIG_VOCAB: {len(SIG_VOCAB):,} terms")
SIG_VOCAB.head()

### DTM 

In [ ]:
DTM = (tok[tok['term_str'].isin(SIG_VOCAB.index)]
    .groupby(DOC_LEVEL + ['term_str'])
    .size()
    .unstack('term_str')
    .fillna(0)
    .astype(int)
)
print(f"DTM shape: {DTM.shape}")
DTM.head()

In [ ]:
DTM.to_parquet('DTM.parquet')

### TF-IDF

In [ ]:
TF    = DTM.div(DTM.sum(axis=1), axis=0)
IDF   = VOCAB.loc[SIG_VOCAB.index, 'idf']
TFIDF = TF.multiply(IDF, axis=1)

print(f"TF-IDF shape: {TFIDF.shape}")
TFIDF.head()

In [ ]:
TFIDF.to_parquet('TFIDF.parquet')

In [ ]:
# BOW: long-format sparse count table indexed by (bag + term_str)
BOW = (
    DTM
    .stack()
    .rename("n")
    .to_frame()
)
BOW = BOW[BOW["n"] > 0]

# Add tfidf values
tfidf_long = TFIDF.stack().rename("tfidf")
BOW["tfidf"] = tfidf_long

BOW.to_csv("BOW.csv")
print(f"BOW shape: {len(BOW):,} rows")
print(f"Index: {BOW.index.names}")
BOW.head()

### Normalized TF-IDF (L2)

In [ ]:
from sklearn.preprocessing import normalize

TFIDF_NORM = pd.DataFrame(
    normalize(TFIDF.values, norm='l2'),
    index=TFIDF.index,
    columns=TFIDF.columns
)
print(f"Normalized TF-IDF shape: {TFIDF_NORM.shape}")
TFIDF_NORM.head()

In [ ]:
TFIDF_NORM.to_parquet('TFIDF_NORM.parquet')

### Reduced TF-IDF (SVD / LSA)

In [ ]:
from sklearn.decomposition import TruncatedSVD

svd = TruncatedSVD(n_components=N_COMPONENTS, random_state=42)

TFIDF_REDUCED = pd.DataFrame(
    svd.fit_transform(TFIDF_NORM),
    index=TFIDF_NORM.index,
    columns=[f'PC{i+1}' for i in range(N_COMPONENTS)]
)

print(f"Reduced TF-IDF shape: {TFIDF_REDUCED.shape}")
print(f"Variance explained: {svd.explained_variance_ratio_.sum():.2%}")
TFIDF_REDUCED.head()

In [ ]:
TFIDF_REDUCED.to_csv('TFIDF_REDUCED.csv')